In [ ]:
import sys
sys.path.insert(0, '/content/lung-nodule-fusion-xai')
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/lung_nodule_processed/labels.csv')


In [ ]:
# Extract features for one nodule (smoke test)
import numpy as np
from src.radiomics.extraction import extract_features_from_arrays

row = df.iloc[0]
patch = np.load(row['patch_path'])
mask = np.load(row['mask_path'])

feats = extract_features_from_arrays(
    patch, mask,
    params_yaml='/content/lung-nodule-fusion-xai/configs/radiomics_params.yaml'
)
print(f"Extracted {len(feats)} features")
print(list(feats.keys())[:5])


In [ ]:
# Full extraction (will take ~hours on Colab — save to Drive)
# NOTE: Run this cell once, results cached to parquet
from src.radiomics.extraction import extract_dataset_features

features_df = extract_dataset_features(
    df,
    params_yaml='/content/lung-nodule-fusion-xai/configs/radiomics_params.yaml',
    output_parquet='/content/drive/MyDrive/lung_nodule_processed/radiomic_features.parquet',
    force_rebuild=False,
)
print(f"Feature matrix shape: {features_df.shape}")


In [ ]:
# Feature selection pipeline (train fold 0 only, as demonstration)
from src.radiomics.feature_selection import full_feature_selection_pipeline
import numpy as np

train_mask = (features_df['fold'] != 0).values
result = full_feature_selection_pipeline(
    features_df,
    train_mask=train_mask,
    icc_threshold=0.75,
    mrmr_n=50,
    perturbed_df=None,  # skips ICC filter if no perturbed extraction
    seed=42,
)
print(f"Selected {len(result['selected_features'])} features via LASSO")
print(result['selected_features'][:10])
